In [11]:
import pandas as pd
fake_news=pd.read_csv('C:/project/News_Dataset/Fake.csv')
true_news=pd.read_csv('C:/project/News_Dataset/True.csv')
print("Fake news dataset:")
print(fake_news.head())
print("True news dataset:")
print(true_news.head())






Fake news dataset:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  
True news dataset:
                                               title  \
0  As U.S. budget fight looms, Republicans fli

In [12]:
fake_news['label']=0
true_news['label']=1
all_news=pd.concat([fake_news,true_news],ignore_index= True)
all_news=all_news.sample(frac=1).reset_index(drop=True)
all_news=all_news.drop(['title','subject','date'],axis=1)
print(all_news.head())



                                                text  label
0  Patrick Henningsen 21st Century WireAny busine...      0
1  And this woman is going to try and keep up wit...      0
2  According to insider, and author Edward Klein,...      0
3  Tune in to the Alternate Current Radio Network...      0
4  This is NOT good at all since a guy was caught...      0


In [13]:
import re 
import string 

def clean_text(text): 
    text=text.lower()
    text=re.sub(r'\[.*?\]', '', text)
    text=re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text=re.sub('[\n]','',text)
    return text
all_news['text']=all_news['text'].apply(clean_text)
print(all_news.head())


                                                text  label
0  patrick henningsen 21st century wireany busine...      0
1  and this woman is going to try and keep up wit...      0
2  according to insider and author edward klein h...      0
3  tune in to the alternate current radio network...      0
4  this is not good at all since a guy was caught...      0


In [14]:
from numpy import vectorize
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.model_selection import train_test_split 
x= all_news['text']
y= all_news['label']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
vectorizer =TfidfVectorizer()
xv_train=vectorizer.fit_transform(x_train)
xv_test=vectorizer.transform(x_test)
print("trained and transformed into numbers")


trained and transformed into numbers


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , classification_report
model= LogisticRegression()
model.fit(xv_train,y_train)
prediction=model.predict(xv_test)
score=accuracy_score(y_test,prediction)
print(f"model accuracy: {score*100:.2f}%")
print("\n detailed report:")
print(classification_report(y_test,prediction))

model accuracy: 98.78%

 detailed report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4647
           1       0.99      0.99      0.99      4333

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [16]:
# Create a function to test any news article or headline you want!
def manual_testing(news):
    # 1. Put the raw text into a format Pandas can read
    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)

    # 2. Clean the text using the exact same function we built on Day 1
    new_def_test["text"] = new_def_test["text"].apply(clean_text)
    new_x_test = new_def_test["text"]

    # 3. Convert the cleaned text into numbers using our vectorizer
    new_xv_test = vectorizer.transform(new_x_test)

    # 4. Ask our trained model to make a prediction
    prediction = model.predict(new_xv_test)

    # 5. Output the result clearly
    if prediction[0] == 0:
        return "Prediction: Fake News 🛑"
    else:
        return "Prediction: True News ✅"

# Let's test it out! 
# You can change the text inside the quotes to anything you want.
my_news = "Scientists have discovered a new planet made entirely of solid gold."

result = manual_testing(my_news)

print("\n--- Manual Testing Result ---")
print(f"Article: {my_news}")
print(result)


--- Manual Testing Result ---
Article: Scientists have discovered a new planet made entirely of solid gold.
Prediction: Fake News 🛑
